# Algorithms 20: Hodgkin-Huxley Model of Action Potentials**Learning Objectives:**1. Translate massive systems of biological ODEs into Python code2. Handle time-dependent stimulus functions $s(t)$3. Simulate a neuron firing an Action Potential (Spike)**Prerequisites:** Algorithms 18**Estimated time:** 120 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapter 20

In [ ]:
# Google Colab Setupimport mathimport matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### The Biophysics of a Neuron> 📖 *From Algorithms Ch. 20:* In 1952, Alan Hodgkin and Andrew Huxley published a differential equation model of “action potentials” in the voltage of neurons, winning a Nobel Prize.The current $I$ entering a neuron consists of:- $s(t)$: External stimulus (like an electrode)- $I_{Na}$: Sodium channel flux (increases voltage)- $I_{K}$: Potassium channel flux (decreases voltage)- $I_{L}$: Leakage currentThe voltage equation is: $\frac{dV}{dt} = \frac{1}{C}[s(t) - I_{Na} - I_{K} - I_{L}]$The channel currents depend on gating variables $n$ (potassium activation), $m$ (sodium activation), and $h$ (sodium inactivation), which themselves are governed by their own differential equations:$\frac{dn}{dt} = \alpha_n(V)(1 - n) - \beta_n(V)n$The $\alpha$ and $\beta$ functions are complex exponentials derived experimentally.This is a 4-variable nonlinear system: State is $(V, n, m, h)$!

### Comprehension Check ✅1. Why are $n$, $m$, and $h$ restricted between 0 and 1?2. Notice that the external stimulus is a function of time, $s(t)$. If $s(t) = 0$, what should the neuron's voltage do?<details><summary>💡 Answers</summary>1. They represent the *probability* that a certain ion channel gate is open. $0 = 0\%$ open, $1 = 100\%$ open.2. It should stay at its resting potential ($V=0$ in our shifted coordinate system). If bumped slightly, the leakage current $I_L$ will drag it back to 0.</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: The $\alpha$ and $\beta$ FunctionsLet's write out the mathematical functions for the Potassium ($n$) gate.$\alpha_n(V) = 0.01(10 - V) / (\exp(0.1(10 - V)) - 1)$$\beta_n(V) = 0.125 \exp(-V / 80)$*(Careful! If $V=10$, $\alpha_n$ divides by zero! Use `V = 10.0001` or L'Hopital's limit in your code)*

In [ ]:
import mathdef alpha_n(V):    if abs(10 - V) < 1e-6: V += 1e-6 # prevent division by zero    return 0.01 * (10 - V) / (math.exp(0.1 * (10 - V)) - 1)def beta_n(V):    return 0.125 * math.exp(-V / 80)# print(alpha_n(0), beta_n(0))

---## Phase 1 — Algorithm Construction### Step 1: Implementing the rest of the constantsWrite the $\alpha$ and $\beta$ functions for $m$ and $h$, and define the constants.$\alpha_m(V) = 0.1(25 - V) / (\exp(0.1(25 - V)) - 1)$$\beta_m(V) = 4 \exp(-V / 18)$$\alpha_h(V) = 0.07 \exp(-V / 20)$$\beta_h(V) = 1 / (\exp(0.1(30 - V)) + 1)$

In [ ]:
# CONSTANTSC = 1.0V_Na = 115V_K = -12V_L = 10.6g_Na_bar = 120g_K_bar = 36g_L_bar = 0.3# YOUR CODE HERE to define alpha_m, beta_m, alpha_h, beta_hpass

### Step 2: The Time-Dependent StimulusWe will inject current pulses at specific time intervals to trigger spikes. Write $s(t)$ which returns 150 if $t$ is inside certain intervals, and 0 otherwise.The intervals are: `[10, 11], [20, 21], [30, 40], [50, 51], [53, 54], [56, 57], [59, 60], [62, 63], [65, 66]`.

In [ ]:
def s_t(t):    intervals = [(10,11), (20,21), (30,40), (50,51), (53,54), (56,57), (59,60), (62,63), (65,66)]    for start, end in intervals:        if start <= t <= end:            return 150.0    return 0.0

### Step 3: The 4 Differential EquationsState is `(V, n, m, h)`. Write `dV_dt`, `dn_dt`, `dm_dt`, `dh_dt`.Recall:$I_{Na} = g_{Na} \cdot m^3 h \cdot (V - V_{Na})$$I_{K} = g_{K} \cdot n^4 \cdot (V - V_{K})$$I_{L} = g_{L} \cdot (V - V_{L})$$dV/dt = (s(t) - I_{Na} - I_{K} - I_{L}) / C$

In [ ]:
# def dV_dt(t, state):#     V, n, m, h = state#     I_Na = g_Na_bar * (m**3) * h * (V - V_Na)#     I_K = g_K_bar * (n**4) * (V - V_K)#     I_L = g_L_bar * (V - V_L)#     return (s_t(t) - I_Na - I_K - I_L) / C# def dn_dt(t, state):#     V, n, m, h = state#     return alpha_n(V)*(1 - n) - beta_n(V)*n# WRITE dm_dt and dh_dt# YOUR CODE HERE

---## Phase 2 — Experimental Verification### Firing the NeuronBring over `MultiEulerEstimator`. Initial state at $t=0$ is $V = 0$, and the asymptotic limits for $n, m, h$: $n_0 = \alpha_n(0) / (\alpha_n(0) + \beta_n(0))$Step from 0 to 80 ms with $\Delta t = 0.01$ ms. Plot $V(t)$.

In [ ]:
# PASTE MultiEulerEstimator HERE# Calculate initial state# n0 = alpha_n(0) / (alpha_n(0) + beta_n(0))# m0 = alpha_m(0) / (alpha_m(0) + beta_m(0))# h0 = alpha_h(0) / (alpha_h(0) + beta_h(0))# initial_state = (0.0, n0, m0, h0)# multi_est = MultiEulerEstimator([dV_dt, dn_dt, dm_dt, dh_dt])# pts = multi_est.calc_estimated_points((0.0, initial_state), 0.01, 80.0)# ts = [p[0] for p in pts]# Vs = [p[1][0] for p in pts]# plt.plot(ts, Vs, color='black')# plt.title("Hodgkin-Huxley Action Potentials")# plt.xlabel("Time (ms)"); plt.ylabel("Voltage (mV)"); plt.grid(True)# plt.show()

### 🔍 ReflectionNotice the long stimulus from $t=30$ to $40$ causes a train of multiple rapid spikes! This is how your nervous system encodes high-intensity stimuli: not by larger spikes (spikes are all-or-nothing), but by a higher *frequency* of spikes.---📚 **Next:** Algorithms 21 (Hash Tables)